In [ ]:
# SAFIR: matrix-level five-fold CV only; each fold has an independent ensemble.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import re
import json
import copy
import random
import hashlib
import platform
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
# 1. Paths and fixed paper protocol
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

COMBO_FILE = DATA_DIR / "O'Neil" / "O'Neil.csv"

DRUG_RAW_FEATURE_FILE = DATA_DIR / "O'Neil" / "Graphlet_features_6_raw.csv"

CRISPR_RAW_FEATURE_FILE = DATA_DIR / "Shared" / "CRISPRGeneEffect.csv"

MODEL_FILE = DATA_DIR / "Shared" / "Model.csv"

EXPERIMENT_BLOCK_COL = None

RESULT_ROOT_DIR = PROJECT_DIR / "results" / "SAFIR_5CV"
RESULT_DIR = RESULT_ROOT_DIR

RANDOM_STATE = 42
N_FOLDS = 5
VAL_FRACTION_OF_REMAINING = 0.15
CRISPR_TOP_N = 1000
CRISPR_MIN_VALID_FRACTION = 0.80
CV_STD_DDOF = 1

DEFAULT_TRAIN_OBSERVED_K_LIST = [0, 2, 4, 4, 6, 6, 8, 8, 10, 12]
TRAIN_OBSERVED_K_LIST = list(DEFAULT_TRAIN_OBSERVED_K_LIST)

GRID_SIZE_A = 4
GRID_SIZE_B = 4
N_GRID = GRID_SIZE_A * GRID_SIZE_B
REQUIRE_COMPLETE_4X4 = True

NUM_EPOCHS = 120
BATCH_SIZE = 512
LR = 5e-4
WEIGHT_DECAY = 1e-5
PATIENCE = 20
MIN_DELTA = 1e-6
GRAD_CLIP_NORM = 5.0
NUM_WORKERS = 0

DRUG_SWAP_AUGMENTATION = True
DRUG_SWAP_PROB = 0.50
LAMBDA_OBS_RECON = 0.50
LAMBDA_DELTA = 0.12
LAMBDA_RESIDUAL_CONSISTENCY = 0.10

DRUG_EMB_DIM = 128
CELL_EMB_DIM = 128
GLOBAL_DIM = 256
OBS_EMB_DIM = 64
HIDDEN_DIM = 256
DROPOUT = 0.10
GRID_GNN_LAYERS = 1
GRID_EDGE_EMB_DIM = 16
LOCAL_OBS_EMB_DIM = 32

SAVE_MATCHED_POINTS = True
SAVE_BLOCK_TABLE = True
SAVE_PREDICTIONS = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# 2. One full model; no single-split/cold-start/ablation reporting.
MODEL_MODE_NAME = "ordered_fusion_swapaug"
MODEL_CONFIG = {
    "desc": "Full No-GNN SAFIR; ordered fusion; training-only drug swap; "
            "reference-residual prediction; paper-aligned five-fold CV",
    "use_grid_gnn": False,
    "use_local_obs": True,
    "use_residual_anchor": True,
    "cell_feature_type": "CRISPR_Top1000_FoldTrainOnly",
}

In [ ]:
# 3. Paper O'Neil acquisition protocol
ACTIVE_ENSEMBLE_SIZE = 3
ACTIVE_ENSEMBLE_SEEDS = [42, 2024, 3407]
ACTIVE_INITIAL_SEED_MODE = "random2"
ACTIVE_BATCH_SIZE = 2
ACTIVE_TOTAL_BUDGETS = [0, 2, 4, 6, 8, 10, 12, 14]
ACTIVE_REPEATS = 3
ACTIVE_STRATEGIES = ["uncertainty"]
MAXIMIZE_RESPONSE = True
SAVE_ACTIVE_TRAJECTORY = True

print("Device:", DEVICE)
print("Protocol: 5 test folds; validation = 15% of the other 4 folds")
print("Ensemble members per fold:", ACTIVE_ENSEMBLE_SIZE)
print("Combination responses:", COMBO_FILE)
print("Raw Graphlet counts:", DRUG_RAW_FEATURE_FILE)
print("Raw CRISPR Gene Effect:", CRISPR_RAW_FEATURE_FILE)

In [ ]:
# ============================================================
# 2. Utilities
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


def normalize_match_name(x):
    """

    MK-2206 <-> MK2206
    SN-38 <-> SN38
    Bortezomib <-> BORTEZOMIB
    """
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = re.sub(r"[\s\-_./()]+", "", x)
    return x


def make_block_key(drug_a, drug_b, cell, experiment_block=""):
    pair = sorted([normalize_match_name(drug_a), normalize_match_name(drug_b)])
    return json.dumps([str(experiment_block), *pair, normalize_match_name(cell)],
                      ensure_ascii=False, separators=(",", ":"))


def build_normalized_index_map(index_values, label="feature"):
    key_to_raw = {}
    duplicated = []

    for raw in index_values:
        key = normalize_match_name(raw)
        if key == "":
            continue
        if key in key_to_raw:
            duplicated.append((key, key_to_raw[key], raw))
        else:
            key_to_raw[key] = raw

    if len(duplicated) > 0:
        raise ValueError(f"Ambiguous normalized {label} names: {duplicated[:10]}")

    return key_to_raw


def find_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    return None


def safe_pcc(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2:
        return np.nan
    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def safe_spearman(y_true, y_pred):
    true_rank = pd.Series(np.asarray(y_true, dtype=float)).rank(method="average").to_numpy()
    pred_rank = pd.Series(np.asarray(y_pred, dtype=float)).rank(method="average").to_numpy()
    return safe_pcc(true_rank, pred_rank)


def concordance_index(y_true, y_pred):
    """
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    n = 0
    h_sum = 0.0

    for i in range(len(y_true)):
        for j in range(i + 1, len(y_true)):
            if y_true[i] == y_true[j]:
                continue
            n += 1
            true_order = y_true[i] - y_true[j]
            pred_order = y_pred[i] - y_pred[j]

            if pred_order == 0:
                h_sum += 0.5
            elif true_order * pred_order > 0:
                h_sum += 1.0

    if n == 0:
        return np.nan

    return h_sum / n


def adjacent_delta_corr(y_true_surface, y_pred_surface):
    yt = np.asarray(y_true_surface, dtype=float)
    yp = np.asarray(y_pred_surface, dtype=float)

    true_deltas = []
    pred_deltas = []

    true_deltas.extend((yt[1:, :] - yt[:-1, :]).ravel().tolist())
    true_deltas.extend((yt[:, 1:] - yt[:, :-1]).ravel().tolist())

    pred_deltas.extend((yp[1:, :] - yp[:-1, :]).ravel().tolist())
    pred_deltas.extend((yp[:, 1:] - yp[:, :-1]).ravel().tolist())

    return safe_pcc(true_deltas, pred_deltas)


def topk_hit_metrics(y_true_surface, y_pred_surface, k=3):
    yt = np.asarray(y_true_surface).reshape(-1)
    yp = np.asarray(y_pred_surface).reshape(-1)

    true_best = int(np.argmax(yt))
    pred_order = np.argsort(-yp)

    top1_hit = int(pred_order[0] == true_best)
    topk_hit = int(true_best in pred_order[:k])

    true_best_ij = np.unravel_index(true_best, (GRID_SIZE_A, GRID_SIZE_B))
    pred_best_ij = np.unravel_index(int(pred_order[0]), (GRID_SIZE_A, GRID_SIZE_B))
    best_dose_distance = float(
        abs(true_best_ij[0] - pred_best_ij[0]) + abs(true_best_ij[1] - pred_best_ij[1])
    )

    return {
        "top1_hit": top1_hit,
        f"top{k}_hit": topk_hit,
        "best_dose_distance": best_dose_distance,
        "true_best_flat": true_best,
        "pred_best_flat": int(pred_order[0]),
    }


def make_observed_mask(n_grid, k, rng):
    mask = np.zeros(n_grid, dtype=np.float32)
    if k <= 0:
        return mask
    k = min(k, n_grid)
    idx = rng.choice(n_grid, size=k, replace=False)
    mask[idx] = 1.0
    return mask


def surface_smoothness_np(surface):
    s = np.asarray(surface, dtype=float)
    vals = []
    vals.append(np.mean((s[1:, :] - s[:-1, :]) ** 2))
    vals.append(np.mean((s[:, 1:] - s[:, :-1]) ** 2))
    return float(np.mean(vals))

In [ ]:
def normalize_name(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())


def load_model_alias_table(path):
    df = pd.read_csv(path)
    cols = {normalize_name(c): c for c in df.columns}

    model_id_col = None
    for c in ["modelid", "depmapid", "depmapid"]:
        if c in cols:
            model_id_col = cols[c]
            break

    if model_id_col is None:
        raise ValueError("Model.csv must contain ModelID or DepMap_ID column.")

    alias_cols = []
    for c in df.columns:
        n = normalize_name(c)
        if any(k in n for k in [
            "modelline",
            "modelname",
            "cellline",
            "strippedcelllinename",
            "alias"
        ]):
            alias_cols.append(c)

    if not alias_cols:
        raise ValueError("Model.csv must contain cell-line name or alias columns.")

    rows = []
    for _, r in df.iterrows():
        model_id = str(r[model_id_col]).strip()
        if not model_id or model_id.lower() == "nan":
            continue
        for c in alias_cols:
            v = str(r[c]).strip()
            if v and v.lower() != "nan":
                rows.append((normalize_name(v), model_id))

    alias_map = {}
    for name, model_id in rows:
        if name in alias_map and alias_map[name] != model_id:
            continue
        alias_map[name] = model_id

    return alias_map


def load_raw_feature_table(path, kind):
    df = pd.read_csv(path)

    if kind == "drug":
        name_col = find_column(df, ["name", "drug", "drug_name", "DrugName"])
        if name_col is None:
            name_col = df.columns[0]
        ignored = {"smile", "smiles", "mol", "molecule", "pubchem_id", "cid"}

        cols = [
            c for c in df.columns
            if c != name_col
            and c.lower() not in ignored
            and not c.lower().startswith("unnamed:")
        ]

        numeric = df[cols].apply(pd.to_numeric, errors="coerce")
        if (df[cols].notna() & numeric.isna()).any().any():
            raise ValueError("Invalid drug feature columns.")

        numeric.index = df[name_col].astype(str).str.strip()

        if numeric.isna().any().any() or (numeric < 0).any().any():
            raise ValueError("Graphlet input must be raw nonnegative counts.")

        return numeric.astype(np.float64)

    if kind == "crispr":
        model_map = load_model_alias_table(MODEL_FILE)

        id_col = None
        for c in df.columns:
            if normalize_name(c) in ["modelid", "depmapid"]:
                id_col = c
                break

        if id_col is None:
            raise ValueError("CRISPRGeneEffect.csv must contain ModelID or DepMap_ID.")

        numeric_cols = [
            c for c in df.columns
            if c != id_col
            and not c.lower().startswith("unnamed:")
        ]

        numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

        if numeric.isna().all(axis=1).any():
            raise ValueError("Invalid CRISPR rows.")

        numeric.index = df[id_col].astype(str).str.strip()

        return numeric.astype(np.float64)

    raise ValueError("Unknown feature type.")


def match_cell_to_depmap(cell_name, alias_map):
    key = normalize_name(cell_name)
    if key not in alias_map:
        raise KeyError(f"Unmatched cell line: {cell_name}")
    return alias_map[key]


def prepare_crispr_by_cells(cell_names, raw_crispr):
    alias_map = load_model_alias_table(MODEL_FILE)

    model_ids = {
        cell: match_cell_to_depmap(cell, alias_map)
        for cell in cell_names
    }

    matched = []
    for cell in cell_names:
        depmap_id = model_ids[cell]
        if depmap_id not in raw_crispr.index:
            raise KeyError(f"Missing DepMap ID in CRISPRGeneEffect: {depmap_id}")
        row = raw_crispr.loc[depmap_id].copy()
        row.name = cell
        matched.append(row)

    return pd.DataFrame(matched)


In [ ]:
# ============================================================
# 4. Match point-level combination data
# ============================================================

def preprocess_combo_points(combo_df, drug_features, cell_features):
    drug1_col = find_column(combo_df, ["drugA_name", "drug1_name", "drug_row", "Drug1", "drug1"])
    drug2_col = find_column(combo_df, ["drugB_name", "drug2_name", "drug_col", "Drug2", "drug2"])
    cell_col = find_column(combo_df, ["cell_line", "cell", "CellLine", "CELL_LINE"])

    conc1_col = find_column(combo_df, ["drugA_conc", "drug1_conc", "conc1", "dose1", "Drug1.Conc"])
    conc2_col = find_column(combo_df, ["drugB_conc", "drug2_conc", "conc2", "dose2", "Drug2.Conc"])

    single1_col = find_column(combo_df, ["single_resp_1", "single_response_1", "singleA", "YA"])
    single2_col = find_column(combo_df, ["single_resp_2", "single_response_2", "singleB", "YB"])

    target_col = find_column(combo_df, ["target", "response", "Y", "X/X0", "inhibition"])

    if EXPERIMENT_BLOCK_COL is not None:
        if EXPERIMENT_BLOCK_COL not in combo_df.columns:
            raise ValueError(f"Assay block column not found: {EXPERIMENT_BLOCK_COL}")
        experiment_col = EXPERIMENT_BLOCK_COL
    else:
        experiment_col = find_column(combo_df, [
            "block_id", "experiment_block_id", "experiment_block", "assay_id", "matrix_id", "block"
        ])
    if experiment_col is not None and combo_df[experiment_col].isna().any():
        raise ValueError("Missing experimental block identifiers; cannot safely construct complete matrices.")
    print("Experimental block column:", experiment_col or "absent (one existing matrix per drug pair/cell required)")

    required = [drug1_col, drug2_col, cell_col, conc1_col, conc2_col, single1_col, single2_col, target_col]
    if any(c is None for c in required):
        raise ValueError(
            + str(combo_df.columns.tolist())
        )

    drug_key_to_raw = build_normalized_index_map(drug_features.index, label="drug_features")
    cell_key_to_raw = build_normalized_index_map(cell_features.index, label="cell_features")

    combo_drugs = sorted(
        set(combo_df[drug1_col].dropna().astype(str)) |
        set(combo_df[drug2_col].dropna().astype(str))
    )
    exact_drug_match = len(set(combo_drugs) & set(map(str, drug_features.index)))
    norm_drug_match = sum(1 for d in combo_drugs if normalize_match_name(d) in drug_key_to_raw)

    print("\n========== Drug name matching check ==========")
    print("Combo unique drugs:", len(combo_drugs))
    print("Drug feature rows:", len(drug_features))
    print("Exact matched combo drugs:", exact_drug_match)
    print("Normalized matched combo drugs:", norm_drug_match)

    norm_missing_drugs = [d for d in combo_drugs if normalize_match_name(d) not in drug_key_to_raw]
    if len(norm_missing_drugs) > 0:
        print("[Warning] Still missing drugs after normalized matching:")
        print(norm_missing_drugs)
    else:
        print("All combo drugs matched after normalized matching.")

    rows = []
    skipped = {"drugA": 0, "drugB": 0, "cell": 0, "error": 0}

    for _, row in combo_df.iterrows():
        try:
            drugA = str(row[drug1_col]).strip()
            drugB = str(row[drug2_col]).strip()
            cell = str(row[cell_col]).strip()

            drugA_key = normalize_match_name(drugA)
            drugB_key = normalize_match_name(drugB)
            cell_key = normalize_match_name(cell)

            if drugA_key not in drug_key_to_raw:
                skipped["drugA"] += 1
                continue
            if drugB_key not in drug_key_to_raw:
                skipped["drugB"] += 1
                continue
            if cell_key not in cell_key_to_raw:
                skipped["cell"] += 1
                continue

            drugA_feature_name = drug_key_to_raw[drugA_key]
            drugB_feature_name = drug_key_to_raw[drugB_key]
            cell_feature_name = cell_key_to_raw[cell_key]

            drugA_conc = float(row[conc1_col])
            drugB_conc = float(row[conc2_col])
            single_resp_1 = float(row[single1_col])
            single_resp_2 = float(row[single2_col])
            target = float(row[target_col])

            if not np.isfinite([drugA_conc, drugB_conc, single_resp_1, single_resp_2, target]).all():
                skipped["error"] += 1
                continue

            if drugA_conc < 0 or drugB_conc < 0:
                skipped["error"] += 1
                continue
            # A zero-dose margin is single-agent input, not a joint-dose target.
            if drugA_conc == 0 or drugB_conc == 0:
                continue
            rows.append({
                "experiment_block": str(row[experiment_col]) if experiment_col is not None else "",
                "drugA_name": drugA,
                "drugB_name": drugB,
                "cell_line": cell,
                "drugA_feature_name": drugA_feature_name,
                "drugB_feature_name": drugB_feature_name,
                "cell_feature_name": cell_feature_name,
                "drugA_conc": drugA_conc,
                "drugB_conc": drugB_conc,
                "single_resp_1": single_resp_1,
                "single_resp_2": single_resp_2,
                "target": target,
            })
        except Exception:
            skipped["error"] += 1

    data_df = pd.DataFrame(rows)
    if data_df.empty:
        raise RuntimeError("No finite joint-dose records matched the raw drug/cell feature tables.")
    if experiment_col is None:
        coordinates = ["drugA_name", "drugB_name", "cell_line", "drugA_conc", "drugB_conc"]
        if data_df.duplicated(coordinates, keep=False).any():
            raise ValueError(
                "Repeated dose coordinates without an experimental block ID. Restore the block identifier "
                "instead of averaging responses across independent experiments."
            )

    print("\n========== Matched point-level data ==========")
    print("Valid point samples:", len(data_df))
    print("Valid cell lines:", data_df["cell_line"].nunique() if len(data_df) else 0)
    print("Valid drugA unique:", data_df["drugA_name"].nunique() if len(data_df) else 0)
    print("Valid drugB unique:", data_df["drugB_name"].nunique() if len(data_df) else 0)
    print("Skipped:", skipped)

    if SAVE_MATCHED_POINTS:
        out_path = RESULT_DIR / "matched_point_level_data.csv"
        data_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("Saved matched point-level data:", out_path)

    return data_df

In [ ]:
# ============================================================
# 5. Build block-level 4x4 surface data
# ============================================================

def build_grid_features_from_axes(doseA_levels, doseB_levels, ya, yb):
    """
    Build the exact 17-D point-wise grid features used by SAFIR.

    Keeping this logic in one helper is important for drug-swap augmentation:
    after swapping A/B we REBUILD the grid features from the swapped dose and
    single-agent axes rather than trying to permute/sign-flip individual columns.
    """
    doseA_levels = np.asarray(doseA_levels, dtype=np.float32)
    doseB_levels = np.asarray(doseB_levels, dtype=np.float32)
    ya = np.asarray(ya, dtype=np.float32)
    yb = np.asarray(yb, dtype=np.float32)

    if len(doseA_levels) != GRID_SIZE_A or len(doseB_levels) != GRID_SIZE_B:
        raise ValueError(
            f"Expected {GRID_SIZE_A}x{GRID_SIZE_B} dose axes, got "
            f"{len(doseA_levels)}x{len(doseB_levels)}"
        )

    logA = np.log10(doseA_levels + 1e-12)
    logB = np.log10(doseB_levels + 1e-12)
    logA_norm = (logA - logA.mean()) / (logA.std() + 1e-6)
    logB_norm = (logB - logB.mean()) / (logB.std() + 1e-6)

    grid_feats = []
    for i in range(GRID_SIZE_A):
        for j in range(GRID_SIZE_B):
            a = logA_norm[i]
            b = logB_norm[j]
            yai = ya[i]
            ybj = yb[j]

            bliss_like = yai + ybj - yai * ybj
            max_single = max(yai, ybj)
            min_single = min(yai, ybj)

            grid_feats.append([
                a,
                b,
                a * b,
                abs(a - b),
                a + b,
                yai,
                ybj,
                yai * ybj,
                abs(yai - ybj),
                yai + ybj,
                bliss_like,
                max_single,
                min_single,
                yai - ybj,
                (yai - ybj) ** 2,
                a ** 2,
                b ** 2,
            ])

    return np.asarray(grid_feats, dtype=np.float32)


def transpose_flat_grid(x):
    """Transpose a flattened GRID_SIZE_A x GRID_SIZE_B array.

    This code is for the O'Neil 4x4 experiment, so the transposed shape is the
    same.  An explicit guard prevents silently using it for a non-square grid.
    """
    if GRID_SIZE_A != GRID_SIZE_B:
        raise ValueError(
            "Drug-swap augmentation in this O'Neil script assumes a square grid. "
            f"Current grid is {GRID_SIZE_A}x{GRID_SIZE_B}."
        )
    arr = np.asarray(x)
    return arr.reshape(GRID_SIZE_A, GRID_SIZE_B).T.reshape(-1).copy()


def build_surface_blocks(point_df, drug_features, cell_features):
    """
    - doseA/doseB
    - YA[4], YB[4]
    - Y[4,4]
    - grid_features[16, 17]
    """
    blocks = []
    skipped = {
        "not_4x4": 0,
        "missing_points": 0,
        "feature_error": 0,
        "duplicated_coordinates": 0,
    }

    group_cols = ["experiment_block", "drugA_name", "drugB_name", "cell_line"]
    grouped = point_df.groupby(group_cols, sort=False)

    for (experiment_block, drugA, drugB, cell), g in grouped:
        try:
            doseA_levels = sorted(g["drugA_conc"].unique())
            doseB_levels = sorted(g["drugB_conc"].unique())

            if REQUIRE_COMPLETE_4X4:
                if len(doseA_levels) != GRID_SIZE_A or len(doseB_levels) != GRID_SIZE_B:
                    skipped["not_4x4"] += 1
                    continue

            if g.duplicated(["drugA_conc", "drugB_conc"], keep=False).any():
                skipped["duplicated_coordinates"] += 1
                continue
            y_mat = g.pivot(index="drugA_conc", columns="drugB_conc", values="target").reindex(
                index=doseA_levels, columns=doseB_levels
            )

            if y_mat.isna().any().any():
                skipped["missing_points"] += 1
                continue

            ya = (
                g.groupby("drugA_conc")["single_resp_1"]
                .mean()
                .reindex(doseA_levels)
                .values
                .astype(np.float32)
            )
            yb = (
                g.groupby("drugB_conc")["single_resp_2"]
                .mean()
                .reindex(doseB_levels)
                .values
                .astype(np.float32)
            )

            if np.any(~np.isfinite(ya)) or np.any(~np.isfinite(yb)):
                skipped["missing_points"] += 1
                continue

            # feature names should be constant inside block
            drugA_feature_name = g["drugA_feature_name"].iloc[0]
            drugB_feature_name = g["drugB_feature_name"].iloc[0]
            cell_feature_name = g["cell_feature_name"].iloc[0]

            # Features are attached only AFTER splitting and fitting this fold.
            drugA_feat = np.empty(0, dtype=np.float32)
            drugB_feat = np.empty(0, dtype=np.float32)
            cell_feat = np.empty(0, dtype=np.float32)

            y = y_mat.values.astype(np.float32)

            # Build the canonical A/B-oriented point-wise dose/single-agent features.
            grid_feats = build_grid_features_from_axes(
                doseA_levels=doseA_levels,
                doseB_levels=doseB_levels,
                ya=ya,
                yb=yb,
            )

            blocks.append({
                "block_id": len(blocks),
                "block_key": make_block_key(drugA, drugB, cell, experiment_block),
                "experiment_block": str(experiment_block),
                "drugA_name": drugA,
                "drugB_name": drugB,
                "cell_line": cell,
                "drugA_feature_name": drugA_feature_name,
                "drugB_feature_name": drugB_feature_name,
                "cell_feature_name": cell_feature_name,
                "doseA_levels": np.asarray(doseA_levels, dtype=np.float32),
                "doseB_levels": np.asarray(doseB_levels, dtype=np.float32),
                "YA": ya,
                "YB": yb,
                "Y": y,
                "grid_features": grid_feats,
                "drugA_feat": drugA_feat,
                "drugB_feat": drugB_feat,
                "cell_feat": cell_feat,
            })

        except Exception:
            skipped["feature_error"] += 1

    print("\n========== Block-level surface data ==========")
    print("Valid 4x4 blocks:", len(blocks))
    print("Skipped:", skipped)

    if len(blocks) == 0:
        raise RuntimeError("No valid 4x4 blocks were constructed. Please check the data.")

    blocks.sort(key=lambda b: b["block_key"])
    keys = [b["block_key"] for b in blocks]
    if len(set(keys)) != len(keys):
        raise ValueError("Duplicate physical matrix identities, possibly from both A/B orientations or aliases.")
    for index, block in enumerate(blocks):
        block["block_id"] = index

    block_summary = []
    for b in blocks:
        block_summary.append({
            "block_id": b["block_id"],
            "block_key": b["block_key"],
            "experiment_block": b["experiment_block"],
            "drugA_name": b["drugA_name"],
            "drugB_name": b["drugB_name"],
            "cell_line": b["cell_line"],
            "target_min": float(np.min(b["Y"])),
            "target_max": float(np.max(b["Y"])),
            "target_range": float(np.max(b["Y"]) - np.min(b["Y"])),
            "doseA_levels": "|".join([f"{x:g}" for x in b["doseA_levels"]]),
            "doseB_levels": "|".join([f"{x:g}" for x in b["doseB_levels"]]),
        })

    block_summary_df = pd.DataFrame(block_summary)

    print("Cell lines:", block_summary_df["cell_line"].nunique())
    print("Drug A unique:", block_summary_df["drugA_name"].nunique())
    print("Drug B unique:", block_summary_df["drugB_name"].nunique())
    print("Median target range:", block_summary_df["target_range"].median())

    if SAVE_BLOCK_TABLE:
        out_path = RESULT_DIR / "surface_block_summary.csv"
        block_summary_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("Saved block summary:", out_path)

    return blocks, block_summary_df

In [ ]:
# ============================================================
# 6. Dataset
# ============================================================

class SurfaceBlockDataset(Dataset):
    def __init__(
        self,
        blocks,
        observed_k_list,
        training=True,
        random_state=42,
        drug_swap_augmentation=False,
        drug_swap_prob=0.50,
    ):
        self.blocks = blocks
        self.observed_k_list = list(observed_k_list)
        self.training = bool(training)
        self.drug_swap_augmentation = bool(drug_swap_augmentation)
        self.drug_swap_prob = float(drug_swap_prob)

        if not (0.0 <= self.drug_swap_prob <= 1.0):
            raise ValueError(f"drug_swap_prob must be in [0,1], got {self.drug_swap_prob}")
        if self.drug_swap_augmentation and GRID_SIZE_A != GRID_SIZE_B:
            raise ValueError(
                "This O'Neil drug-swap augmentation implementation requires a square grid; "
                f"got {GRID_SIZE_A}x{GRID_SIZE_B}."
            )

        # Preserve the original mask-K random stream for fair comparison with the
        # no-augmentation model.  Swap decisions use a separate RNG.
        self.rng = np.random.default_rng(random_state)
        self.swap_rng = np.random.default_rng(int(random_state) + 104729)

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, idx):
        b = self.blocks[idx]

        # Random Mask-K is generated exactly as in the original model.
        if self.training:
            k = int(self.rng.choice(self.observed_k_list))
            obs_mask = make_observed_mask(N_GRID, k, self.rng)
        else:
            # Evaluation functions construct their own masks; dataset default is K=0.
            k = 0
            obs_mask = np.zeros(N_GRID, dtype=np.float32)

        # Drug-swap augmentation is TRAINING-ONLY and uses an independent RNG.
        # The same physical experiment is re-expressed as (B,A):
        #   drugA <-> drugB
        #   doseA <-> doseB
        #   YA <-> YB
        #   Y -> Y^T
        #   observation/loss masks -> transpose
        do_swap = (
            self.training
            and self.drug_swap_augmentation
            and (self.swap_rng.random() < self.drug_swap_prob)
        )

        if do_swap:
            drugA_feat = b["drugB_feat"].astype(np.float32)
            drugB_feat = b["drugA_feat"].astype(np.float32)
            YA = b["YB"].astype(np.float32)
            YB = b["YA"].astype(np.float32)
            Y = b["Y"].T.copy().astype(np.float32)

            # Rebuild orientation-dependent grid features after swapping axes.
            grid_features = build_grid_features_from_axes(
                doseA_levels=b["doseB_levels"],
                doseB_levels=b["doseA_levels"],
                ya=YA,
                yb=YB,
            )

            # Preserve the same physical context points after A/B exchange.
            obs_mask = transpose_flat_grid(obs_mask).astype(np.float32)
            loss_mask = transpose_flat_grid(
                b.get("loss_mask", np.ones(N_GRID, dtype=np.float32))
            ).astype(np.float32)
            eval_dose_mask = transpose_flat_grid(
                b.get("eval_dose_mask", np.zeros(N_GRID, dtype=np.float32))
            ).astype(np.float32)
        else:
            drugA_feat = b["drugA_feat"].astype(np.float32)
            drugB_feat = b["drugB_feat"].astype(np.float32)
            YA = b["YA"].astype(np.float32)
            YB = b["YB"].astype(np.float32)
            Y = b["Y"].astype(np.float32)
            grid_features = b["grid_features"].astype(np.float32)
            loss_mask = b.get(
                "loss_mask", np.ones(N_GRID, dtype=np.float32)
            ).astype(np.float32)
            eval_dose_mask = b.get(
                "eval_dose_mask", np.zeros(N_GRID, dtype=np.float32)
            ).astype(np.float32)

        y_flat = Y.reshape(-1).astype(np.float32)
        obs_values = y_flat * obs_mask

        item = {
            "block_id": np.int64(b["block_id"]),
            "drugA_feat": drugA_feat,
            "drugB_feat": drugB_feat,
            "cell_feat": b["cell_feat"].astype(np.float32),
            "grid_features": grid_features,
            "YA": YA,
            "YB": YB,
            "Y": Y,
            "Y_flat": y_flat,
            "obs_mask": obs_mask.astype(np.float32),
            "obs_values": obs_values.astype(np.float32),
            "loss_mask": loss_mask,
            "eval_dose_mask": eval_dose_mask,
            "observed_k": np.int64(k),
            "drug_swapped": np.int64(1 if do_swap else 0),
        }
        return item


def collate_fn(batch):
    out = {}
    keys = batch[0].keys()
    for k in keys:
        vals = [b[k] for b in batch]
        if k in ["block_id", "observed_k", "drug_swapped"]:
            out[k] = torch.tensor(vals, dtype=torch.long)
        else:
            out[k] = torch.tensor(np.stack(vals), dtype=torch.float32)
    return out


def make_loader(
    blocks, observed_k_list, training=True, shuffle=True,
    drug_swap_augmentation=False, drug_swap_prob=DRUG_SWAP_PROB,
):
    ds = SurfaceBlockDataset(
        blocks=blocks,
        observed_k_list=observed_k_list,
        training=training,
        random_state=RANDOM_STATE,
        drug_swap_augmentation=drug_swap_augmentation,
        drug_swap_prob=drug_swap_prob,
    )
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE.type == "cuda" else False,
        persistent_workers=True if NUM_WORKERS > 0 else False,
        collate_fn=collate_fn,
    )


In [ ]:
# ============================================================
# 7. Model: SAFIR-Active Residual-aware GridGNN
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.0, use_bn=True):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class GridMessagePassingLayer(nn.Module):
    """
    Residual-aware masked Grid-GNN layer.

    """
    def __init__(self, hidden_dim, edge_emb_dim, dropout=0.0, num_edge_types=8):
        super().__init__()
        self.edge_emb = nn.Embedding(num_edge_types, edge_emb_dim)

        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim + edge_emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.gate_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_emb_dim + 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

        self.update_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h, edge_index, edge_type, obs_mask):
        """
        h: [B, 16, H]
        edge_index: [2, E]
        edge_type: [E]
        """
        B, N, H = h.shape
        src = edge_index[0]
        dst = edge_index[1]
        E = src.numel()

        h_src = h[:, src, :]                                      # [B, E, H]
        h_dst = h[:, dst, :]                                      # [B, E, H]
        e = self.edge_emb(edge_type).unsqueeze(0).expand(B, E, -1) # [B, E, edge_dim]

        src_obs = obs_mask[:, src].unsqueeze(-1)                  # [B, E, 1]
        dst_obs = obs_mask[:, dst].unsqueeze(-1)                  # [B, E, 1]

        msg_input = torch.cat([h_src, e], dim=-1)
        msg = self.message_mlp(msg_input.reshape(B * E, -1)).reshape(B, E, H)

        gate_input = torch.cat([h_src, h_dst, e, src_obs, dst_obs], dim=-1)
        gate = self.gate_mlp(gate_input.reshape(B * E, -1)).reshape(B, E, 1)

        src_conf = 0.15 + 0.85 * src_obs
        edge_weight = gate * src_conf
        msg = msg * edge_weight

        agg = torch.zeros(B, N, H, device=h.device, dtype=h.dtype)
        agg.index_add_(1, dst, msg)

        deg = torch.zeros(B, N, 1, device=h.device, dtype=h.dtype)
        deg.index_add_(1, dst, edge_weight)
        agg = agg / deg.clamp(min=1.0)

        delta = self.update_mlp(torch.cat([h, agg], dim=-1).reshape(B * N, -1)).reshape(B, N, H)
        h = self.norm1(h + self.dropout(delta))

        f = self.ffn(h.reshape(B * N, H)).reshape(B, N, H)
        h = self.norm2(h + self.dropout(f))
        return h

class SAFIRDoseSurfaceCompletionNet(nn.Module):
    """
    SAFIR-Dose-GridGNN model_mode-aware model.

    """

    def __init__(self, drug_dim, cell_dim, grid_feat_dim,
                 use_grid_gnn=True,
                 use_local_obs=True,
                 use_residual_anchor=True):
        super().__init__()
        self.use_grid_gnn = bool(use_grid_gnn)
        self.use_local_obs = bool(use_local_obs)
        self.use_residual_anchor = bool(use_residual_anchor)

        self.drug_encoder = MLP(
            in_dim=drug_dim,
            hidden_dims=[512, 256],
            out_dim=DRUG_EMB_DIM,
            dropout=DROPOUT,
            use_bn=True,
        )

        self.cell_encoder = MLP(
            in_dim=cell_dim,
            hidden_dims=[512, 256],
            out_dim=CELL_EMB_DIM,
            dropout=DROPOUT,
            use_bn=True,
        )

        global_in_dim = DRUG_EMB_DIM * 4 + CELL_EMB_DIM
        self.global_encoder = MLP(
            in_dim=global_in_dim,
            hidden_dims=[384],
            out_dim=GLOBAL_DIM,
            dropout=DROPOUT,
            use_bn=True,
        )

        self.obs_encoder = MLP(
            in_dim=N_GRID * 3 + 5,
            hidden_dims=[128],
            out_dim=OBS_EMB_DIM,
            dropout=DROPOUT,
            use_bn=True,
        )

        if self.use_local_obs:
            self.local_obs_encoder = nn.Sequential(
                nn.Linear(4, LOCAL_OBS_EMB_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(LOCAL_OBS_EMB_DIM, LOCAL_OBS_EMB_DIM),
                nn.ReLU(),
            )
            local_dim = LOCAL_OBS_EMB_DIM
        else:
            self.local_obs_encoder = None
            local_dim = 0

        if self.use_residual_anchor:
            base_node_in_dim = GLOBAL_DIM + grid_feat_dim
            self.base_node_projector = nn.Sequential(
                nn.Linear(base_node_in_dim, HIDDEN_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
                nn.ReLU(),
            )

            if self.use_grid_gnn:
                eps_node_in_dim = GLOBAL_DIM + grid_feat_dim + local_dim
            else:
                eps_node_in_dim = GLOBAL_DIM + OBS_EMB_DIM + grid_feat_dim + local_dim

            self.eps_node_projector = nn.Sequential(
                nn.Linear(eps_node_in_dim, HIDDEN_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
                nn.ReLU(),
            )

            self.node_projector = None
        else:
            node_in_dim = GLOBAL_DIM + OBS_EMB_DIM + grid_feat_dim + local_dim
            self.node_projector = nn.Sequential(
                nn.Linear(node_in_dim, HIDDEN_DIM),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
                nn.ReLU(),
            )
            self.base_node_projector = None
            self.eps_node_projector = None

        if self.use_grid_gnn:
            self.grid_gnn_layers = nn.ModuleList([
                GridMessagePassingLayer(
                    hidden_dim=HIDDEN_DIM,
                    edge_emb_dim=GRID_EDGE_EMB_DIM,
                    dropout=DROPOUT,
                    num_edge_types=8,
                )
                for _ in range(GRID_GNN_LAYERS)
            ])
        else:
            self.grid_gnn_layers = nn.ModuleList([])

        if self.use_residual_anchor:
            self.ab_head = nn.Linear(HIDDEN_DIM, 2)
            self.epsilon_head = nn.Linear(HIDDEN_DIM, 1)
            self.epsilon_gate = nn.Sequential(
                nn.Linear(OBS_EMB_DIM + GLOBAL_DIM, 128),
                nn.ReLU(),
                nn.Linear(128, 1),
                nn.Sigmoid(),
            )
            self.direct_head = None
        else:
            self.ab_head = None
            self.epsilon_head = None
            self.epsilon_gate = None
            self.direct_head = nn.Linear(HIDDEN_DIM, 1)
        edge_index, edge_type = self.build_grid_edges(GRID_SIZE_A, GRID_SIZE_B)
        self.register_buffer("edge_index", edge_index, persistent=False)
        self.register_buffer("edge_type", edge_type, persistent=False)

    @staticmethod
    def build_grid_edges(n_a, n_b):
        """
        node index = i * n_b + j

        edge type:
        0: drugB increase
        1: drugB decrease
        2: drugA increase
        3: drugA decrease
        4: drugA increase + drugB increase
        5: drugA decrease + drugB decrease
        6: drugA increase + drugB decrease
        7: drugA decrease + drugB increase
        """
        edges = []
        types = []

        def add_directed(u, v, t):
            edges.append((u, v))
            types.append(t)

        for i in range(n_a):
            for j in range(n_b):
                u = i * n_b + j

                # horizontal: drugB direction
                if j + 1 < n_b:
                    v = i * n_b + (j + 1)
                    add_directed(u, v, 0)  # B increase
                    add_directed(v, u, 1)  # B decrease

                # vertical: drugA direction
                if i + 1 < n_a:
                    v = (i + 1) * n_b + j
                    add_directed(u, v, 2)  # A increase
                    add_directed(v, u, 3)  # A decrease

                # diagonal down: both increase/decrease
                if i + 1 < n_a and j + 1 < n_b:
                    v = (i + 1) * n_b + (j + 1)
                    add_directed(u, v, 4)  # A increase, B increase
                    add_directed(v, u, 5)  # A decrease, B decrease

                # diagonal up: one increase, one decrease
                if i + 1 < n_a and j - 1 >= 0:
                    v = (i + 1) * n_b + (j - 1)
                    add_directed(u, v, 6)  # A increase, B decrease
                    add_directed(v, u, 7)  # A decrease, B increase

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_type = torch.tensor(types, dtype=torch.long)
        return edge_index, edge_type
    def encode_observed_points(self, grid_features, obs_values, obs_mask):
        """
        """
        bliss_grid = grid_features[:, :, 10]
        obs_residual = (obs_values - bliss_grid) * obs_mask

        obs_count_raw = obs_mask.sum(dim=1, keepdim=True)
        obs_count = obs_count_raw.clamp(min=1.0)
        obs_mean = (obs_values * obs_mask).sum(dim=1, keepdim=True) / obs_count

        very_neg = torch.full_like(obs_values, -1e9)
        very_pos = torch.full_like(obs_values, 1e9)
        obs_max = torch.where(obs_mask > 0.5, obs_values, very_neg).max(dim=1, keepdim=True).values
        obs_min = torch.where(obs_mask > 0.5, obs_values, very_pos).min(dim=1, keepdim=True).values
        obs_max = torch.where(obs_count_raw > 0, obs_max, torch.zeros_like(obs_max))
        obs_min = torch.where(obs_count_raw > 0, obs_min, torch.zeros_like(obs_min))

        obs_var = (((obs_values - obs_mean) ** 2) * obs_mask).sum(dim=1, keepdim=True) / obs_count
        obs_std = torch.sqrt(obs_var + 1e-8)
        obs_ratio = obs_mask.mean(dim=1, keepdim=True)

        obs_summary = torch.cat([obs_mean, obs_max, obs_min, obs_std, obs_ratio], dim=1)
        obs_input = torch.cat([obs_values, obs_mask, obs_residual, obs_summary], dim=1)
        h_obs = self.obs_encoder(obs_input)

        if self.use_local_obs:
            local_obs_input = torch.stack([
                obs_mask,
                obs_values * obs_mask,
                obs_residual,
                torch.abs(obs_residual),
            ], dim=-1)  # [B, 16, 4]
            B, N, _ = local_obs_input.shape
            h_local_obs = self.local_obs_encoder(local_obs_input.reshape(B * N, -1)).reshape(B, N, -1)
        else:
            h_local_obs = None

        return h_obs, h_local_obs, obs_residual

    def forward(self, batch):
        drugA = batch["drugA_feat"]
        drugB = batch["drugB_feat"]
        cell = batch["cell_feat"]
        grid_features = batch["grid_features"]      # [B, 16, G]
        obs_values = batch["obs_values"]
        obs_mask = batch["obs_mask"]                # [B, 16], observed=1, missing=0

        B = drugA.size(0)

        hA = self.drug_encoder(drugA)
        hB = self.drug_encoder(drugB)
        hC = self.cell_encoder(cell)

        # Keep the ORIGINAL information-rich ordered fusion.
        # Drug-order robustness is encouraged at the DATA level via training-only
        # drug-swap augmentation rather than by deleting hA/hB from this representation.
        drug_pair = torch.cat([
            hA,
            hB,
            hA * hB,
            torch.abs(hA - hB),
            hC,
        ], dim=1)

        h_global = self.global_encoder(drug_pair)  # [B, GLOBAL_DIM]
        h_obs, h_local_obs, obs_residual = self.encode_observed_points(grid_features, obs_values, obs_mask)

        h_global_rep = h_global.unsqueeze(1).expand(B, N_GRID, -1)
        h_obs_rep = h_obs.unsqueeze(1).expand(B, N_GRID, -1)

        if self.use_residual_anchor:
            # ------------------------------------------------------------
            # ------------------------------------------------------------
            base_node_input = torch.cat([h_global_rep, grid_features], dim=-1)
            h_base = self.base_node_projector(
                base_node_input.reshape(B * N_GRID, -1)
            ).reshape(B, N_GRID, HIDDEN_DIM)

            # ------------------------------------------------------------
            # ------------------------------------------------------------
            eps_node_inputs = [h_global_rep, grid_features]
            if not self.use_grid_gnn:
                eps_node_inputs.insert(1, h_obs_rep)
            if self.use_local_obs:
                eps_node_inputs.append(h_local_obs)

            eps_node_input = torch.cat(eps_node_inputs, dim=-1)
            h_eps = self.eps_node_projector(
                eps_node_input.reshape(B * N_GRID, -1)
            ).reshape(B, N_GRID, HIDDEN_DIM)

            if self.use_grid_gnn:
                for layer in self.grid_gnn_layers:
                    h_eps = layer(h_eps, self.edge_index, self.edge_type, obs_mask)

            base_hidden = h_base.reshape(B * N_GRID, HIDDEN_DIM)
            eps_hidden = h_eps.reshape(B * N_GRID, HIDDEN_DIM)

            ab_logits = self.ab_head(base_hidden).reshape(B, N_GRID, 2)
            ab = torch.softmax(ab_logits, dim=-1)
            alpha = ab[:, :, 0]
            beta = ab[:, :, 1]

            epsilon_base = self.epsilon_head(eps_hidden).reshape(B, N_GRID)
            gate_input = torch.cat([h_global, h_obs], dim=1)
            eps_gate = self.epsilon_gate(gate_input)  # [B, 1], 0-1
            epsilon = epsilon_base * (0.5 + 1.5 * eps_gate)

            YA_grid = grid_features[:, :, 5]
            YB_grid = grid_features[:, :, 6]
            base = alpha * YA_grid + beta * YB_grid
            pred = base + epsilon

        else:
            node_inputs = [h_global_rep, h_obs_rep, grid_features]
            if self.use_local_obs:
                node_inputs.append(h_local_obs)
            node_input = torch.cat(node_inputs, dim=-1)
            h = self.node_projector(node_input.reshape(B * N_GRID, -1)).reshape(B, N_GRID, HIDDEN_DIM)

            if self.use_grid_gnn:
                for layer in self.grid_gnn_layers:
                    h = layer(h, self.edge_index, self.edge_type, obs_mask)

            point_hidden = h.reshape(B * N_GRID, HIDDEN_DIM)
            pred = self.direct_head(point_hidden).reshape(B, N_GRID)
            alpha = torch.zeros_like(pred)
            beta = torch.zeros_like(pred)
            base = torch.zeros_like(pred)
            epsilon = torch.zeros_like(pred)
            epsilon_base = torch.zeros_like(pred)

        return {
            "pred_flat": pred,
            "pred_surface": pred.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "epsilon_flat": epsilon,
            "epsilon_surface": epsilon.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "epsilon_base_surface": epsilon_base.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "alpha_surface": alpha.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "beta_surface": beta.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "base_surface": base.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
            "obs_residual_surface": obs_residual.reshape(B, GRID_SIZE_A, GRID_SIZE_B),
        }


In [ ]:
# ============================================================
# 8. Loss
# ============================================================

def delta_loss(pred_surface, true_surface, loss_mask=None):
    """
    """
    dpa = pred_surface[:, 1:, :] - pred_surface[:, :-1, :]
    dta = true_surface[:, 1:, :] - true_surface[:, :-1, :]
    dpb = pred_surface[:, :, 1:] - pred_surface[:, :, :-1]
    dtb = true_surface[:, :, 1:] - true_surface[:, :, :-1]

    if loss_mask is None:
        return (F.mse_loss(dpa, dta) + F.mse_loss(dpb, dtb)) / 2.0

    mask = loss_mask.reshape_as(true_surface)
    ma = mask[:, 1:, :] * mask[:, :-1, :]
    mb = mask[:, :, 1:] * mask[:, :, :-1]

    la = (((dpa - dta) ** 2) * ma).sum() / ma.sum().clamp(min=1.0)
    lb = (((dpb - dtb) ** 2) * mb).sum() / mb.sum().clamp(min=1.0)
    return (la + lb) / 2.0



def compute_loss(outputs, batch):
    """
    Simplified SAFIR objective:
        L = L_reconstruction + lambda_delta * L_delta
            + lambda_residual * L_residual_consistency

    where
        L_reconstruction = L_missing + lambda_obs * L_observed.

    Removed from the original objective:
        - smoothness regularization
        - epsilon L2 regularization
    """
    pred = outputs["pred_flat"]
    y = batch["Y_flat"]
    obs_mask = batch["obs_mask"]
    loss_mask = batch.get("loss_mask", torch.ones_like(obs_mask))

    miss_mask = (1.0 - obs_mask) * loss_mask
    obs_loss_mask = obs_mask * loss_mask

    # 1) Reconstruction loss: missing positions are the primary target;
    #    observed positions receive a lower-weight consistency term.
    miss_count = miss_mask.sum().clamp(min=1.0)
    obs_count = obs_loss_mask.sum().clamp(min=1.0)

    loss_missing = (((pred - y) ** 2) * miss_mask).sum() / miss_count
    loss_obs = (((pred - y) ** 2) * obs_loss_mask).sum() / obs_count
    loss_reconstruction = loss_missing + LAMBDA_OBS_RECON * loss_obs

    # 2) Landscape local-difference consistency along the two dose axes.
    loss_delta = delta_loss(
        outputs["pred_surface"],
        batch["Y"],
        loss_mask=loss_mask,
    )

    # 3) Observation-guided residual consistency.
    #    epsilon should approximate Y - Base at observed positions.
    #    Base is detached so this auxiliary supervision only updates the residual branch.
    base_flat = outputs["base_surface"].reshape_as(y)
    if outputs["epsilon_flat"].requires_grad:
        true_obs_residual = (y - base_flat.detach()) * obs_loss_mask
        loss_residual_consistency = (
            ((outputs["epsilon_flat"] - true_obs_residual) ** 2) * obs_loss_mask
        ).sum() / obs_count
    else:
        loss_residual_consistency = pred.new_tensor(0.0)

    total = (
        loss_reconstruction
        + LAMBDA_DELTA * loss_delta
        + LAMBDA_RESIDUAL_CONSISTENCY * loss_residual_consistency
    )

    return total, {
        "loss_total": float(total.detach().cpu()),
        "loss_reconstruction": float(loss_reconstruction.detach().cpu()),
        "loss_missing": float(loss_missing.detach().cpu()),
        "loss_obs": float(loss_obs.detach().cpu()),
        "loss_delta": float(loss_delta.detach().cpu()),
        "loss_residual_consistency": float(loss_residual_consistency.detach().cpu()),
    }


In [ ]:
# 9. Five mutually exclusive outer test folds, then training-only preprocessing.
def make_fivefold_splits(blocks):
    if N_FOLDS != 5 or not np.isclose(VAL_FRACTION_OF_REMAINING, 0.15):
        raise ValueError("The paper protocol requires 5 folds and validation = 15% of the remaining matrices.")
    if len(blocks) < 10:
        raise ValueError("Too few complete matrices for nonempty five-fold train/validation/test partitions.")
    keys = [b["block_key"] for b in blocks]
    if len(set(keys)) != len(keys):
        raise ValueError("Every complete physical matrix must have one unique block_key.")
    indices = np.arange(len(blocks))
    test_counts = np.zeros(len(blocks), dtype=int)
    splits = []
    splitter = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for fold_id, (remaining_idx, test_idx) in enumerate(splitter.split(indices), start=1):
        train_idx, val_idx = train_test_split(
            remaining_idx, test_size=VAL_FRACTION_OF_REMAINING,
            random_state=RANDOM_STATE + fold_id, shuffle=True,
        )
        groups = [set(train_idx), set(val_idx), set(test_idx)]
        if any(not part for part in groups) or any(groups[i] & groups[j] for i in range(3) for j in range(i)):
            raise RuntimeError(f"Empty or overlapping partitions in fold {fold_id}.")
        if set.union(*groups) != set(indices):
            raise RuntimeError(f"Incomplete matrix assignment in fold {fold_id}.")
        test_counts[test_idx] += 1
        splits.append((fold_id, np.sort(train_idx), np.sort(val_idx), np.sort(test_idx)))
    if not np.all(test_counts == 1):
        raise RuntimeError("Every matrix must appear in exactly one held-out test fold.")
    return splits


def fit_fold_preprocessors(train_blocks, raw_drug_features, raw_crispr):
    train_drugs = sorted({b[k] for b in train_blocks
                          for k in ("drugA_feature_name", "drugB_feature_name")})
    train_cells = sorted({b["cell_feature_name"] for b in train_blocks})
    if len(train_cells) < 2:
        raise ValueError("At least two unique training cells are needed to select variable CRISPR genes.")
    drug_scaler = StandardScaler().fit(raw_drug_features.loc[train_drugs].to_numpy())
    train_cell_table = raw_crispr.loc[train_cells]
    valid = train_cell_table.notna().mean() >= CRISPR_MIN_VALID_FRACTION
    variances = train_cell_table.var(axis=0, ddof=0, skipna=True)
    eligible = valid & np.isfinite(variances) & (variances > 0)
    selected_genes = variances[eligible].sort_values(ascending=False, kind="mergesort").head(CRISPR_TOP_N).index.tolist()
    if len(selected_genes) != CRISPR_TOP_N:
        raise ValueError(
            f"Only {len(selected_genes)} eligible genes in the training cells; "
            f"the protocol needs {CRISPR_TOP_N}. Check the raw CRISPR table."
        )
    train_selected = train_cell_table[selected_genes]
    medians = train_selected.median(axis=0, skipna=True)
    if not np.isfinite(medians.to_numpy()).all():
        raise ValueError("Training-cell medians contain nonfinite values.")
    cell_scaler = StandardScaler().fit(train_selected.fillna(medians).to_numpy())
    return {
        "train_drugs": train_drugs,
        "train_cells": train_cells,
        "drug_columns": raw_drug_features.columns.tolist(),
        "selected_genes": selected_genes,
        "cell_medians": medians,
        "drug_scaler": drug_scaler,
        "cell_scaler": cell_scaler,
        "crispr_min_valid_fraction": CRISPR_MIN_VALID_FRACTION,
    }


def transform_fold_blocks(blocks, raw_drug_features, raw_crispr, preprocessor):
    drug_table = pd.DataFrame(
        preprocessor["drug_scaler"].transform(raw_drug_features[preprocessor["drug_columns"]].to_numpy()),
        index=raw_drug_features.index, columns=preprocessor["drug_columns"],
    )
    selected = raw_crispr[preprocessor["selected_genes"]].fillna(preprocessor["cell_medians"])
    cell_table = pd.DataFrame(preprocessor["cell_scaler"].transform(selected.to_numpy()),
                              index=raw_crispr.index, columns=preprocessor["selected_genes"])
    transformed = []
    for block in blocks:
        item = dict(block)
        item["drugA_feat"] = drug_table.loc[block["drugA_feature_name"]].to_numpy(dtype=np.float32)
        item["drugB_feat"] = drug_table.loc[block["drugB_feature_name"]].to_numpy(dtype=np.float32)
        item["cell_feat"] = cell_table.loc[block["cell_feature_name"]].to_numpy(dtype=np.float32)
        if not all(np.isfinite(item[k]).all() for k in ("drugA_feat", "drugB_feat", "cell_feat")):
            raise RuntimeError("Nonfinite transformed features.")
        transformed.append(item)
    return transformed, drug_table, cell_table


def save_fold_preprocessing(fold_dir, fold_id, preprocessor, drug_table, cell_table):
    joblib.dump(preprocessor, fold_dir / "fold_preprocessors.joblib")
    drug_table.to_csv(fold_dir / "drug_features_fold_train_scaled.csv", index_label="name", encoding="utf-8-sig")
    cell_table.to_csv(fold_dir / "crispr_top1000_fold_train_scaled.csv", index_label="cell_line", encoding="utf-8-sig")
    audit = {
        "fold_id": fold_id,
        "fit_scope": "unique training drugs and unique training cells only",
        "train_drugs": preprocessor["train_drugs"],
        "train_cells": preprocessor["train_cells"],
        "selected_genes": preprocessor["selected_genes"],
        "crispr_min_valid_fraction": CRISPR_MIN_VALID_FRACTION,
        "validation_and_test": "transform only; no feature fitting, model selection or parameter updates on test",
    }
    (fold_dir / "preprocessing_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
# 10. Validation and reporting: repeated observations are NOT independent folds.
@torch.no_grad()
def evaluate_val_loss(model, val_loader):
    model.eval()
    # Reuse identical validation masks at every epoch for stable checkpoint selection.
    val_loader.dataset.rng = np.random.default_rng(val_loader.dataset.evaluation_mask_seed)
    losses = []
    sizes = []
    for batch in val_loader:
        sizes.append(int(batch["Y_flat"].shape[0]))
        batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
        loss, _ = compute_loss(model(batch), batch)
        losses.append(float(loss.detach().cpu()))
    if not losses or not np.isfinite(losses).all():
        raise RuntimeError("Empty/nonfinite validation loss; cannot select a checkpoint.")
    return float(np.average(losses, weights=sizes))


PRIMARY_METRICS = ["rmse_all", "mae_all", "pcc_all", "spearman_all"]
MISSING_METRICS = ["rmse_missing", "mae_missing", "pcc_missing", "spearman_missing"]


def summarize_test_fold(active_df, fold_id, expected_test_ids):
    ids = set(map(int, expected_test_ids))
    if set(active_df["block_id"].astype(int)) != ids:
        raise RuntimeError("Fold metrics do not match the held-out matrices.")
    if set(active_df["strategy"]) != {"uncertainty"}:
        raise RuntimeError("Only the paper uncertainty strategy is reportable here.")
    if set(active_df["budget"].astype(int)) != set(ACTIVE_TOTAL_BUDGETS):
        raise RuntimeError("Fold is missing an evaluation budget.")
    keys = ["block_id", "repeat", "strategy", "budget"]
    if active_df.duplicated(keys).any():
        raise RuntimeError("Duplicate block/repeat/budget metrics would bias the fold results.")
    expected_rows = len(ids) * ACTIVE_REPEATS * len(ACTIVE_TOTAL_BUDGETS)
    counts = active_df.groupby(["block_id", "strategy", "budget"]).size()
    if len(active_df) != expected_rows or not (counts == ACTIVE_REPEATS).all():
        raise RuntimeError("Each held-out matrix/budget must have all three acquisition repeats.")
    repeat_sets = active_df.groupby(["block_id", "strategy", "budget"])["repeat"].agg(lambda s: set(s))
    if not all(value == set(range(ACTIVE_REPEATS)) for value in repeat_sets):
        raise RuntimeError("Acquisition repeat identifiers are incomplete.")
    metric_cols = PRIMARY_METRICS + MISSING_METRICS
    by_matrix = active_df.groupby(["strategy", "budget", "block_id"])[metric_cols].mean().reset_index()
    if not np.isfinite(by_matrix[["rmse_all", "mae_all"]].to_numpy()).all():
        raise RuntimeError("Nonfinite full-landscape error metrics.")
    grouped = by_matrix.groupby(["strategy", "budget"])
    fold = grouped[metric_cols].mean().reset_index()
    for metric in metric_cols:
        fold[metric + "_valid_matrices"] = grouped[metric].count().to_numpy()
    fold["fold_id"] = int(fold_id)
    fold["test_matrices"] = len(ids)
    fold["acquisition_repeats"] = ACTIVE_REPEATS
    return fold.sort_values("budget").reset_index(drop=True)


def summarize_five_test_folds(fold_metrics):
    expected_folds = set(range(1, 6))
    if set(fold_metrics["fold_id"].astype(int)) != expected_folds:
        raise RuntimeError("A final result requires all five held-out test folds.")
    if fold_metrics.duplicated(["fold_id", "strategy", "budget"]).any():
        raise RuntimeError("More than one metric row for a test fold/budget.")
    if set(fold_metrics["strategy"]) != {"uncertainty"} or set(fold_metrics["budget"]) != set(ACTIVE_TOTAL_BUDGETS):
        raise RuntimeError("Final table does not match the paper strategy/budgets.")
    rows = []
    for (strategy, budget), group in fold_metrics.groupby(["strategy", "budget"], sort=True):
        if len(group) != 5 or set(group["fold_id"]) != expected_folds:
            raise RuntimeError(f"Missing held-out folds at budget {budget}.")
        row = {"strategy": strategy, "budget": int(budget), "n_test_folds": 5,
               "std_ddof": CV_STD_DDOF, "metric_scope": "full_landscape",
               "within_fold_aggregation": "repeat_mean_then_matrix_mean"}
        for metric in PRIMARY_METRICS:
            values = group[metric].to_numpy(dtype=float)
            if not np.isfinite(values).all():
                raise RuntimeError(f"{metric} is not finite in all five test folds at K={budget}.")
            row[metric + "_mean"] = float(values.mean())
            row[metric + "_std"] = float(values.std(ddof=CV_STD_DDOF))
        rows.append(row)
    numeric = pd.DataFrame(rows).sort_values("budget").reset_index(drop=True)
    formatted = pd.DataFrame({"K": numeric["budget"]})
    for label, metric in [("RMSE", "rmse_all"), ("MAE", "mae_all"),
                          ("PCC", "pcc_all"), ("Spearman", "spearman_all")]:
        formatted[label] = [f"{mean:.4f} ± {std:.4f}" for mean, std in
                            zip(numeric[metric + "_mean"], numeric[metric + "_std"])]
    return numeric, formatted

In [ ]:
# ============================================================
# 11. SAFIR-Active: training + dynamic dose-point selection
# ============================================================


def make_loader_with_seed(
    blocks, observed_k_list, training=True, shuffle=True, seed=42,
    drug_swap_augmentation=False, drug_swap_prob=DRUG_SWAP_PROB,
):
    """Loader with independent mask/swap RNG streams for each ensemble seed."""
    ds = SurfaceBlockDataset(
        blocks=blocks,
        observed_k_list=observed_k_list,
        training=training,
        random_state=seed,
        drug_swap_augmentation=drug_swap_augmentation,
        drug_swap_prob=drug_swap_prob,
    )
    ds.evaluation_mask_seed = int(seed)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(int(seed)),
        drop_last=bool(training and shuffle and len(blocks) % BATCH_SIZE == 1),
        num_workers=NUM_WORKERS,
        pin_memory=True if DEVICE.type == "cuda" else False,
        persistent_workers=True if NUM_WORKERS > 0 else False,
        collate_fn=collate_fn,
    )


def block_to_batch_with_mask(block, obs_mask):
    """Inference receives only already revealed target responses, never full hidden Y."""
    obs_mask = np.asarray(obs_mask, dtype=np.float32)
    obs_values = np.where(obs_mask > 0.5, block["Y"].reshape(-1), 0.0).astype(np.float32)
    arrays = {"drugA_feat": block["drugA_feat"], "drugB_feat": block["drugB_feat"],
              "cell_feat": block["cell_feat"], "grid_features": block["grid_features"],
              "obs_mask": obs_mask, "obs_values": obs_values}
    return {key: torch.tensor(value[None, ...], dtype=torch.float32, device=DEVICE)
            for key, value in arrays.items()}


@torch.no_grad()
def ensemble_predict_flat(models, block, obs_mask):
    preds = []
    batch = block_to_batch_with_mask(block, obs_mask)
    for model in models:
        model.eval()
        out = model(batch)
        pred = out["pred_flat"][0].detach().cpu().numpy().astype(np.float32)
        preds.append(pred)
    preds = np.stack(preds, axis=0)  # [E, 16]
    pred_mean = preds.mean(axis=0)
    pred_std = preds.std(axis=0)
    return pred_mean, pred_std, preds



@torch.no_grad()
def ensemble_predict_components_flat(models, block, obs_mask):
    """


    epsilon/base/synergy-related components are retained only for downstream diagnostics
    and do not participate in uncertainty acquisition.
    """
    preds = []
    epsilons = []
    bases = []
    batch = block_to_batch_with_mask(block, obs_mask)
    for model in models:
        model.eval()
        out = model(batch)
        preds.append(out["pred_flat"][0].detach().cpu().numpy().astype(np.float32))
        epsilons.append(out["epsilon_flat"][0].detach().cpu().numpy().astype(np.float32))
        bases.append(out["base_surface"][0].detach().cpu().numpy().reshape(-1).astype(np.float32))

    preds = np.stack(preds, axis=0)
    epsilons = np.stack(epsilons, axis=0)
    bases = np.stack(bases, axis=0)

    pred_mean = preds.mean(axis=0)
    pred_std = preds.std(axis=0)
    epsilon_mean = epsilons.mean(axis=0)
    epsilon_std = epsilons.std(axis=0)
    base_mean = bases.mean(axis=0)
    base_std = bases.std(axis=0)

    bliss = np.asarray(block["grid_features"][:, 10], dtype=np.float32)
    synergy_preds = preds - bliss[None, :]
    synergy_mean = synergy_preds.mean(axis=0)
    synergy_std = synergy_preds.std(axis=0)

    return {
        "pred_mean": pred_mean,
        "pred_std": pred_std,
        "preds": preds,
        "epsilon_mean": epsilon_mean,
        "epsilon_std": epsilon_std,
        "epsilons": epsilons,
        "base_mean": base_mean,
        "base_std": base_std,
        "synergy_mean": synergy_mean,
        "synergy_std": synergy_std,
        "synergy_preds": synergy_preds,
        "bliss": bliss,
    }

def initial_observed_mask(mode, rng):
    mask = np.zeros(N_GRID, dtype=np.float32)
    if mode is None or mode == "none":
        return mask
    if mode == "random2":
        idx = rng.choice(N_GRID, size=min(2, N_GRID), replace=False)
        mask[idx] = 1.0
        return mask
    if mode == "corners4":
        idx = [0, GRID_SIZE_B - 1, (GRID_SIZE_A - 1) * GRID_SIZE_B, N_GRID - 1]
        mask[idx] = 1.0
        return mask
    if mode == "center4":
        idx = [1 * GRID_SIZE_B + 1, 1 * GRID_SIZE_B + 2, 2 * GRID_SIZE_B + 1, 2 * GRID_SIZE_B + 2]
        mask[idx] = 1.0
        return mask
    raise ValueError(f"Unknown ACTIVE_INITIAL_SEED_MODE: {mode}")











def surface_shape_diagnostics(y_true_surface, y_pred_surface):
    """
    """
    yt = np.asarray(y_true_surface, dtype=float).reshape(GRID_SIZE_A, GRID_SIZE_B)
    yp = np.asarray(y_pred_surface, dtype=float).reshape(GRID_SIZE_A, GRID_SIZE_B)

    true_grad = []
    pred_grad = []
    true_grad.extend((yt[1:, :] - yt[:-1, :]).reshape(-1).tolist())
    true_grad.extend((yt[:, 1:] - yt[:, :-1]).reshape(-1).tolist())
    pred_grad.extend((yp[1:, :] - yp[:-1, :]).reshape(-1).tolist())
    pred_grad.extend((yp[:, 1:] - yp[:, :-1]).reshape(-1).tolist())
    true_grad = np.asarray(true_grad, dtype=float)
    pred_grad = np.asarray(pred_grad, dtype=float)

    grad_rmse = float(np.sqrt(mean_squared_error(true_grad, pred_grad))) if len(true_grad) > 0 else np.nan
    grad_mae = float(mean_absolute_error(true_grad, pred_grad)) if len(true_grad) > 0 else np.nan
    grad_corr = safe_pcc(true_grad, pred_grad)

    def laplacian_like(arr):
        vals = []
        for i in range(GRID_SIZE_A):
            for j in range(GRID_SIZE_B):
                neigh = []
                for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    ni, nj = i + di, j + dj
                    if 0 <= ni < GRID_SIZE_A and 0 <= nj < GRID_SIZE_B:
                        neigh.append(arr[ni, nj])
                if len(neigh) > 0:
                    vals.append(arr[i, j] - np.mean(neigh))
                else:
                    vals.append(0.0)
        return np.asarray(vals, dtype=float)

    true_lap = laplacian_like(yt)
    pred_lap = laplacian_like(yp)
    lap_rmse = float(np.sqrt(mean_squared_error(true_lap, pred_lap)))
    lap_mae = float(mean_absolute_error(true_lap, pred_lap))
    lap_corr = safe_pcc(true_lap, pred_lap)

    return {
        "surface_gradient_rmse": grad_rmse,
        "surface_gradient_mae": grad_mae,
        "surface_gradient_corr": grad_corr,
        "surface_laplacian_rmse": lap_rmse,
        "surface_laplacian_mae": lap_mae,
        "surface_laplacian_corr": lap_corr,
    }



def select_next_points(
    strategy,
    pred_mean,
    pred_std,
    obs_mask,
    batch_size,
    rng,
    epsilon_mean=None,
    epsilon_std=None,
    synergy_mean=None,
    synergy_std=None,
):
    """Uncertainty-only acquisition: select unobserved positions with largest ensemble std."""
    if strategy != "uncertainty":
        raise ValueError(
            f"This notebook is uncertainty-only, but got strategy={strategy!r}. "
            "Set ACTIVE_STRATEGIES = ['uncertainty']."
        )

    unobs_idx = np.where(obs_mask < 0.5)[0]
    if len(unobs_idx) == 0:
        return []

    batch_size = min(int(batch_size), len(unobs_idx))
    score = np.asarray(pred_std, dtype=float).copy()
    score[obs_mask > 0.5] = -np.inf
    return np.argsort(-score)[:batch_size].tolist()


def true_top_indices(y_flat, k=3):
    if MAXIMIZE_RESPONSE:
        return np.argsort(-y_flat)[:k]
    return np.argsort(y_flat)[:k]


def flat_to_ij(idx):
    return int(idx // GRID_SIZE_B), int(idx % GRID_SIZE_B)


def _top_region_metrics(true_score, pred_score, obs_mask, prefix, larger_is_better=True, topk=3):
    true_score = np.asarray(true_score, dtype=float)
    pred_score = np.asarray(pred_score, dtype=float)
    obs_mask = np.asarray(obs_mask, dtype=np.float32)

    if larger_is_better:
        true_order = np.argsort(-true_score)
        pred_order = np.argsort(-pred_score)
    else:
        true_order = np.argsort(true_score)
        pred_order = np.argsort(pred_score)

    true_best = int(true_order[0])
    pred_best = int(pred_order[0])
    true_topk = set(true_order[:topk].tolist())
    pred_topk = set(pred_order[:topk].tolist())

    ti, tj = flat_to_ij(true_best)
    pi, pj = flat_to_ij(pred_best)
    true_best_value = float(true_score[true_best])
    pred_best_true_value = float(true_score[pred_best])
    regret = float(true_best_value - pred_best_true_value) if larger_is_better else float(pred_best_true_value - true_best_value)

    observed_idx = np.where(obs_mask > 0.5)[0]
    if len(observed_idx) > 0:
        observed_set = set(observed_idx.tolist())
        observed_top1_hit = float(true_best in observed_set)
        observed_top3_hit = float(len(true_topk & observed_set) > 0)
        if larger_is_better:
            obs_best_idx = int(observed_idx[np.argmax(true_score[observed_idx])])
            obs_best_value = float(np.max(true_score[observed_idx]))
            observed_regret = float(true_best_value - obs_best_value)
        else:
            obs_best_idx = int(observed_idx[np.argmin(true_score[observed_idx])])
            obs_best_value = float(np.min(true_score[observed_idx]))
            observed_regret = float(obs_best_value - true_best_value)
        oi, oj = flat_to_ij(obs_best_idx)
        observed_best_distance = float(abs(ti - oi) + abs(tj - oj))
    else:
        observed_top1_hit = 0.0
        observed_top3_hit = 0.0
        obs_best_idx = -1
        obs_best_value = np.nan
        observed_regret = np.nan
        observed_best_distance = np.nan

    q = max(1, int(np.ceil(0.25 * len(true_score))))
    true_hot = set(true_order[:q].tolist())
    pred_hot = set(pred_order[:q].tolist())
    obs_hot = set(observed_idx.tolist()) if len(observed_idx) > 0 else set()
    pred_jaccard = float(len(true_hot & pred_hot) / max(1, len(true_hot | pred_hot)))
    observed_recall = float(len(true_hot & obs_hot) / max(1, len(true_hot)))

    return {
        f"{prefix}_pred_top1_hit": float(pred_best == true_best),
        f"{prefix}_pred_top3_hit": float(len(true_topk & pred_topk) > 0),
        f"{prefix}_pred_best_distance": float(abs(ti - pi) + abs(tj - pj)),
        f"{prefix}_pred_regret": regret,
        f"{prefix}_observed_top1_hit": observed_top1_hit,
        f"{prefix}_observed_top3_hit": observed_top3_hit,
        f"{prefix}_observed_best_distance": observed_best_distance,
        f"{prefix}_observed_regret": observed_regret,
        f"{prefix}_true_best_flat": true_best,
        f"{prefix}_pred_best_flat": pred_best,
        f"{prefix}_observed_best_flat": obs_best_idx,
        f"{prefix}_true_best_value": true_best_value,
        f"{prefix}_pred_best_true_value": pred_best_true_value,
        f"{prefix}_observed_best_true_value": obs_best_value,
        f"{prefix}_top25_jaccard_pred": pred_jaccard,
        f"{prefix}_top25_recall_observed": observed_recall,
    }


def evaluate_active_state(
    block,
    pred_mean,
    pred_std,
    obs_mask,
    epsilon_mean=None,
    epsilon_std=None,
    synergy_mean=None,
    synergy_std=None,
):
    y_flat = block["Y"].reshape(-1).astype(float)
    pred_flat = np.asarray(pred_mean, dtype=float)
    pred_std = np.asarray(pred_std, dtype=float)
    obs_mask = np.asarray(obs_mask, dtype=np.float32)
    miss_mask = 1.0 - obs_mask

    rmse_all = float(np.sqrt(mean_squared_error(y_flat, pred_flat)))
    mae_all = float(mean_absolute_error(y_flat, pred_flat))
    pcc_all = safe_pcc(y_flat, pred_flat)
    cindex_all = concordance_index(y_flat, pred_flat)
    spearman_all = safe_spearman(y_flat, pred_flat)

    if miss_mask.sum() > 0:
        miss_bool = miss_mask.astype(bool)
        rmse_missing = float(np.sqrt(mean_squared_error(y_flat[miss_bool], pred_flat[miss_bool])))
        mae_missing = float(mean_absolute_error(y_flat[miss_bool], pred_flat[miss_bool]))
        pcc_missing = safe_pcc(y_flat[miss_bool], pred_flat[miss_bool])
        spearman_missing = safe_spearman(y_flat[miss_bool], pred_flat[miss_bool])
    else:
        rmse_missing = np.nan
        mae_missing = np.nan
        pcc_missing = np.nan
        spearman_missing = np.nan

    response_metrics = _top_region_metrics(
        true_score=y_flat,
        pred_score=pred_flat,
        obs_mask=obs_mask,
        prefix="response",
        larger_is_better=MAXIMIZE_RESPONSE,
        topk=3,
    )

    pred_top1_hit = response_metrics["response_pred_top1_hit"]
    pred_top3_hit = response_metrics["response_pred_top3_hit"]
    pred_best_distance = response_metrics["response_pred_best_distance"]
    pred_regret = response_metrics["response_pred_regret"]
    observed_top1_hit = response_metrics["response_observed_top1_hit"]
    observed_top3_hit = response_metrics["response_observed_top3_hit"]
    observed_best_distance = response_metrics["response_observed_best_distance"]
    observed_regret = response_metrics["response_observed_regret"]
    true_best = response_metrics["response_true_best_flat"]
    pred_best = response_metrics["response_pred_best_flat"]
    obs_best_idx = response_metrics["response_observed_best_flat"]
    true_best_value = response_metrics["response_true_best_value"]
    pred_best_true_value = response_metrics["response_pred_best_true_value"]
    obs_best_value = response_metrics["response_observed_best_true_value"]

    pred_surface = pred_flat.reshape(GRID_SIZE_A, GRID_SIZE_B)
    surface_shape_metrics = surface_shape_diagnostics(block["Y"], pred_surface)

    bliss = np.asarray(block["grid_features"][:, 10], dtype=float)
    true_synergy = y_flat - bliss
    if synergy_mean is None:
        pred_synergy = pred_flat - bliss
    else:
        pred_synergy = np.asarray(synergy_mean, dtype=float)
    synergy_metrics = _top_region_metrics(
        true_score=true_synergy,
        pred_score=pred_synergy,
        obs_mask=obs_mask,
        prefix="synergy",
        larger_is_better=True,
        topk=3,
    )

    # residual/evidence summary
    epsilon_mean_arr = np.asarray(epsilon_mean, dtype=float) if epsilon_mean is not None else np.full_like(pred_flat, np.nan, dtype=float)
    epsilon_std_arr = np.asarray(epsilon_std, dtype=float) if epsilon_std is not None else np.full_like(pred_flat, np.nan, dtype=float)
    synergy_std_arr = np.asarray(synergy_std, dtype=float) if synergy_std is not None else np.full_like(pred_flat, np.nan, dtype=float)

    if miss_mask.sum() > 0:
        miss_bool = miss_mask.astype(bool)
        mean_epsilon_unobserved = float(np.nanmean(epsilon_mean_arr[miss_bool]))
        mean_positive_epsilon_unobserved = float(np.nanmean(np.maximum(epsilon_mean_arr[miss_bool], 0.0)))
        mean_abs_epsilon_unobserved = float(np.nanmean(np.abs(epsilon_mean_arr[miss_bool])))
        mean_synergy_unobserved = float(np.nanmean(pred_synergy[miss_bool]))
        mean_positive_synergy_unobserved = float(np.nanmean(np.maximum(pred_synergy[miss_bool], 0.0)))
        mean_synergy_uncertainty_unobserved = float(np.nanmean(synergy_std_arr[miss_bool]))
    else:
        mean_epsilon_unobserved = 0.0
        mean_positive_epsilon_unobserved = 0.0
        mean_abs_epsilon_unobserved = 0.0
        mean_synergy_unobserved = 0.0
        mean_positive_synergy_unobserved = 0.0
        mean_synergy_uncertainty_unobserved = 0.0


    out = {
        "rmse_all": rmse_all,
        "mae_all": mae_all,
        "pcc_all": pcc_all,
        "spearman_all": spearman_all,
        "c_index_all": cindex_all,
        "rmse_missing": rmse_missing,
        "mae_missing": mae_missing,
        "pcc_missing": pcc_missing,
        "spearman_missing": spearman_missing,
        "adjacent_delta_corr": adjacent_delta_corr(block["Y"], pred_surface),
        "pred_top1_hit": pred_top1_hit,
        "pred_top3_hit": pred_top3_hit,
        "pred_best_distance": pred_best_distance,
        "pred_regret": pred_regret,
        "observed_top1_hit": observed_top1_hit,
        "observed_top3_hit": observed_top3_hit,
        "observed_best_distance": observed_best_distance,
        "observed_regret": observed_regret,
        "true_best_flat": true_best,
        "pred_best_flat": pred_best,
        "observed_best_flat": obs_best_idx,
        "true_best_value": true_best_value,
        "pred_best_true_value": pred_best_true_value,
        "observed_best_true_value": obs_best_value,
        "mean_uncertainty_unobserved": float(np.mean(pred_std[miss_mask.astype(bool)])) if miss_mask.sum() > 0 else 0.0,
        "mean_epsilon_unobserved": mean_epsilon_unobserved,
        "mean_positive_epsilon_unobserved": mean_positive_epsilon_unobserved,
        "mean_abs_epsilon_unobserved": mean_abs_epsilon_unobserved,
        "mean_synergy_unobserved": mean_synergy_unobserved,
        "mean_positive_synergy_unobserved": mean_positive_synergy_unobserved,
        "mean_synergy_uncertainty_unobserved": mean_synergy_uncertainty_unobserved,
    }
    out.update(response_metrics)
    out.update(synergy_metrics)
    out.update(surface_shape_metrics)
    return out

def simulate_active_one_block(models, block, strategy, repeat_seed, total_budgets):
    """One sequential trajectory, plus its zero-observation baseline."""
    budgets = sorted(set(map(int, total_budgets)))
    if any(k < 0 or k > N_GRID for k in budgets) or not budgets:
        raise ValueError("Invalid observation budgets.")
    rng = np.random.default_rng(repeat_seed)
    rows, trajectories, predictions = [], [], []
    obs_mask = np.zeros(N_GRID, dtype=np.float32)
    selected_history = []
    round_id = 0

    def record_state(comp):
        k = int(obs_mask.sum())
        row = {"block_id": block["block_id"], "block_key": block["block_key"],
               "experiment_block": block.get("experiment_block", ""),
               "drugA_name": block["drugA_name"], "drugB_name": block["drugB_name"],
               "cell_line": block["cell_line"], "strategy": strategy, "repeat_seed": repeat_seed,
               "budget": k, "round": round_id,
               "selected_history": ";".join(map(str, selected_history))}
        row.update(evaluate_active_state(block, comp["pred_mean"], comp["pred_std"], obs_mask,
                                        comp["epsilon_mean"], comp["epsilon_std"],
                                        comp["synergy_mean"], comp["synergy_std"]))
        rows.append(row)
        if SAVE_PREDICTIONS:
            for idx in range(N_GRID):
                i, j = flat_to_ij(idx)
                predictions.append({"block_id": block["block_id"], "block_key": block["block_key"],
                                    "strategy": strategy, "repeat_seed": repeat_seed, "budget": k,
                                    "flat_index": idx, "doseA_index": i, "doseB_index": j,
                                    "is_observed_input": int(obs_mask[idx]),
                                    "target": float(block["Y"].reshape(-1)[idx]),
                                    "prediction": float(comp["pred_mean"][idx]),
                                    "uncertainty": float(comp["pred_std"][idx])})

    zero_comp = ensemble_predict_components_flat(models, block, obs_mask)
    if 0 in budgets:
        record_state(zero_comp)
    initial_mask = initial_observed_mask(ACTIVE_INITIAL_SEED_MODE, rng)
    initial_indices = np.flatnonzero(initial_mask > 0.5).tolist()
    initial_k = len(initial_indices)
    if any(0 < k < initial_k for k in budgets):
        raise ValueError("A requested nonzero budget is below the initial observation count.")
    if max(budgets) > 0 and max(budgets) < initial_k:
        raise ValueError("Initial observations exceed the maximum requested budget.")

    def reveal(indices, comp, selection_kind):
        for idx in indices:
            if obs_mask[idx] > 0.5:
                raise RuntimeError("An observed position was selected again.")
            obs_mask[idx] = 1.0
            selected_history.append(int(idx))
            if SAVE_ACTIVE_TRAJECTORY:
                i, j = flat_to_ij(idx)
                trajectories.append({"block_id": block["block_id"], "block_key": block["block_key"],
                                     "strategy": strategy, "repeat_seed": repeat_seed, "round": round_id,
                                     "selection_kind": selection_kind, "selected_flat": int(idx),
                                     "doseA_index": i, "doseB_index": j,
                                     "true_response": float(block["Y"].reshape(-1)[idx]),
                                     "pred_response_before_measurement": float(comp["pred_mean"][idx]),
                                     "pred_uncertainty_before_measurement": float(comp["pred_std"][idx]),
                                     "observed_count_after": int(obs_mask.sum())})

    if max(budgets) > 0:
        reveal(initial_indices, zero_comp, "random_initial")
    comp = zero_comp if max(budgets) == 0 else ensemble_predict_components_flat(models, block, obs_mask)
    if initial_k in budgets and initial_k > 0:
        record_state(comp)
    while int(obs_mask.sum()) < max(budgets):
        current = int(obs_mask.sum())
        next_budget = min(k for k in budgets if k > current)
        need = min(ACTIVE_BATCH_SIZE, next_budget - current)
        next_points = select_next_points(strategy, comp["pred_mean"], comp["pred_std"],
                                         obs_mask, need, rng)
        if len(next_points) != need:
            raise RuntimeError("Acquisition did not return the required unobserved positions.")
        round_id += 1
        reveal(next_points, comp, "uncertainty_acquisition")
        # Fixed-parameter re-inference after every acquisition batch.
        comp = ensemble_predict_components_flat(models, block, obs_mask)
        if int(obs_mask.sum()) in budgets:
            record_state(comp)
    if [row["budget"] for row in rows] != budgets:
        raise RuntimeError("Not all requested states were evaluated exactly once.")
    return rows, trajectories, predictions


def run_active_simulation(models, test_blocks, fold_dir, fold_id):
    active_rows, trajectories, predictions = [], [], []
    for repeat in range(ACTIVE_REPEATS):
        for block in test_blocks:
            token = f"{RANDOM_STATE}|{fold_id}|{repeat}|{block['block_key']}"
            repeat_seed = int.from_bytes(hashlib.sha256(token.encode("utf-8")).digest()[:8], "little")
            rows, traj, pred = simulate_active_one_block(models, block, "uncertainty", repeat_seed, ACTIVE_TOTAL_BUDGETS)
            for collection in (rows, traj, pred):
                for row in collection:
                    row["fold_id"] = fold_id
                    row["repeat"] = repeat
            active_rows.extend(rows)
            trajectories.extend(traj)
            predictions.extend(pred)
    active_df = pd.DataFrame(active_rows)
    active_df.to_csv(fold_dir / "test_matrix_metrics.csv", index=False, encoding="utf-8-sig")
    if SAVE_ACTIVE_TRAJECTORY:
        pd.DataFrame(trajectories).to_csv(fold_dir / "test_acquisition_trajectory.csv", index=False, encoding="utf-8-sig")
    if SAVE_PREDICTIONS:
        pd.DataFrame(predictions).to_csv(fold_dir / "test_dose_predictions.csv", index=False, encoding="utf-8-sig")
    fold_metrics = summarize_test_fold(active_df, fold_id, [b["block_id"] for b in test_blocks])
    fold_metrics.to_csv(fold_dir / "test_fold_metrics.csv", index=False, encoding="utf-8-sig")
    return fold_metrics


def train_single_model_for_ensemble(tr_blocks, val_blocks, setting_dir, model_mode_config, seed, ensemble_id):
    seed_everything(seed)
    train_loader = make_loader_with_seed(
        tr_blocks,
        TRAIN_OBSERVED_K_LIST,
        training=True,
        shuffle=True,
        seed=seed,
        drug_swap_augmentation=DRUG_SWAP_AUGMENTATION,
        drug_swap_prob=DRUG_SWAP_PROB,
    )
    # Validation uses fixed-seed Random Mask-K; no swap augmentation.
    val_loader = make_loader_with_seed(
        val_blocks,
        TRAIN_OBSERVED_K_LIST,
        training=True,
        shuffle=False,
        seed=seed + 17,
        drug_swap_augmentation=False,
        drug_swap_prob=0.0,
    )

    drug_dim = tr_blocks[0]["drugA_feat"].shape[0]
    cell_dim = tr_blocks[0]["cell_feat"].shape[0]
    grid_feat_dim = tr_blocks[0]["grid_features"].shape[1]

    model = SAFIRDoseSurfaceCompletionNet(
        drug_dim=drug_dim,
        cell_dim=cell_dim,
        grid_feat_dim=grid_feat_dim,
        use_grid_gnn=model_mode_config["use_grid_gnn"],
        use_local_obs=model_mode_config["use_local_obs"],
        use_residual_anchor=model_mode_config["use_residual_anchor"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
    )

    best_val = float("inf")
    best_epoch = None
    best_state = None
    epochs_no_improve = 0
    history = []

    print(f"\n--- Train ensemble member {ensemble_id} | seed={seed} ---")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        epoch_losses = []
        for batch in train_loader:
            batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(batch)
            loss, loss_dict = compute_loss(out, batch)
            loss.backward()
            if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            epoch_losses.append(loss_dict)

        train_loss = float(np.mean([x["loss_total"] for x in epoch_losses]))
        train_recon = float(np.mean([x["loss_reconstruction"] for x in epoch_losses]))
        train_missing = float(np.mean([x["loss_missing"] for x in epoch_losses]))
        train_obs = float(np.mean([x["loss_obs"] for x in epoch_losses]))
        train_delta = float(np.mean([x["loss_delta"] for x in epoch_losses]))
        train_residual = float(np.mean([x["loss_residual_consistency"] for x in epoch_losses]))
        val_loss = evaluate_val_loss(model, val_loader)
        scheduler.step(val_loss)

        history.append({
            "ensemble_id": ensemble_id,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_reconstruction_loss": train_recon,
            "train_missing_loss": train_missing,
            "train_observed_loss": train_obs,
            "train_delta_loss": train_delta,
            "train_residual_consistency_loss": train_residual,
            "val_loss": val_loss,
            "lr": optimizer.param_groups[0]["lr"],
        })

        print(
            f"Ensemble {ensemble_id} | Epoch {epoch:03d}/{NUM_EPOCHS} | "
            f"Train {train_loss:.5f} | Recon {train_recon:.5f} | "
            f"Delta {train_delta:.5f} | Residual {train_residual:.5f} | Val {val_loss:.5f}"
        )

        if val_loss < best_val - MIN_DELTA:
            best_val = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            torch.save(best_state, setting_dir / f"best_model_ensemble{ensemble_id}_seed{seed}.pt")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping ensemble {ensemble_id} at epoch {epoch}. Best epoch={best_epoch}, val={best_val:.6f}")
            break

    pd.DataFrame(history).to_csv(
        setting_dir / f"training_history_ensemble{ensemble_id}_seed{seed}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    if best_state is None:
        raise RuntimeError("No finite validation checkpoint was produced; test evaluation is forbidden.")
    model.load_state_dict(best_state)
    model.eval()
    return model, {"ensemble_id": ensemble_id, "seed": seed, "best_epoch": best_epoch, "best_val_loss": best_val}


def train_one_test_fold(train_blocks, val_blocks, test_blocks, fold_dir, fold_id):
    """Each fold is a separate train/validation/test experiment with E=3 models."""
    print(f"\nFold {fold_id}/5: train={len(train_blocks)}, val={len(val_blocks)}, test={len(test_blocks)}")
    seeds = [int(seed) + fold_id * 1009 for seed in ACTIVE_ENSEMBLE_SEEDS]
    models, info_rows = [], []
    for ensemble_id, seed in enumerate(seeds):
        model, info = train_single_model_for_ensemble(train_blocks, val_blocks, fold_dir,
                                                       MODEL_CONFIG, seed, ensemble_id)
        info["fold_id"] = fold_id
        models.append(model)
        info_rows.append(info)
    pd.DataFrame(info_rows).to_csv(fold_dir / "ensemble_training_summary.csv", index=False, encoding="utf-8-sig")
    # Held-out test responses enter only the permitted sparse observation state.
    metrics = run_active_simulation(models, test_blocks, fold_dir, fold_id)
    del models
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return metrics

In [ ]:
# 12. Execute exactly five test folds and publish only their aggregate.
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_paper_protocol():
    if N_FOLDS != 5 or not np.isclose(VAL_FRACTION_OF_REMAINING, 0.15) or CV_STD_DDOF != 1:
        raise ValueError("Required: five folds, 15% validation within the remaining data, sample SD (ddof=1).")
    if (ACTIVE_ENSEMBLE_SIZE != 3 or len(ACTIVE_ENSEMBLE_SEEDS) != 3
            or len(set(ACTIVE_ENSEMBLE_SEEDS)) != 3):
        raise ValueError("Each fold needs exactly three independently seeded ensemble members.")
    if ACTIVE_INITIAL_SEED_MODE != "random2" or ACTIVE_BATCH_SIZE != 2 or ACTIVE_REPEATS != 3:
        raise ValueError("Required acquisition: random2 start, batches of 2, three initial-observation repeats.")
    if ACTIVE_STRATEGIES != ["uncertainty"] or ACTIVE_TOTAL_BUDGETS != list(range(0, 15, 2)):
        raise ValueError("Only uncertainty acquisition at K=0,2,...,14 is reported.")
    if N_GRID != 16 or GRID_SIZE_A != 4 or GRID_SIZE_B != 4:
        raise ValueError("This notebook implements the O'Neil 4x4 main experiment.")
    if CRISPR_TOP_N != 1000 or not np.isclose(CRISPR_MIN_VALID_FRACTION, 0.80):
        raise ValueError("Paper CRISPR protocol: >=80% training-cell coverage, Top1000 variable genes.")
    if TRAIN_OBSERVED_K_LIST != DEFAULT_TRAIN_OBSERVED_K_LIST:
        raise ValueError("Restore the paper Random Mask-K probabilities before running.")
    if NUM_WORKERS != 0 or BATCH_SIZE < 2:
        raise ValueError("Use NUM_WORKERS=0 and BATCH_SIZE>=2 for reproducible mask streams and BatchNorm.")


def main():
    global RESULT_DIR
    validate_paper_protocol()
    seed_everything(RANDOM_STATE)
    # A separate directory prevents an interrupted run from exposing an old final table.
    run_id = datetime.now(timezone.utc).strftime("cv_%Y%m%d_%H%M%S_%f")
    RESULT_DIR = RESULT_ROOT_DIR / run_id
    RESULT_DIR.mkdir(parents=True, exist_ok=False)
    combo_df, raw_drug, raw_crispr = load_raw_data()
    point_df = preprocess_combo_points(combo_df, raw_drug, raw_crispr)
    blocks, _ = build_surface_blocks(point_df, raw_drug, raw_crispr)
    splits = make_fivefold_splits(blocks)
    split_rows = []
    for fold_id, train_idx, val_idx, test_idx in splits:
        for split_name, indices in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
            for index in indices:
                block = blocks[int(index)]
                split_rows.append({"fold_id": fold_id, "split": split_name,
                                   "block_id": block["block_id"], "block_key": block["block_key"],
                                   "experiment_block": block["experiment_block"],
                                   "drugA_name": block["drugA_name"], "drugB_name": block["drugB_name"],
                                   "cell_line": block["cell_line"]})
    pd.DataFrame(split_rows).to_csv(RESULT_DIR / "fivefold_matrix_splits.csv", index=False, encoding="utf-8-sig")
    configuration = {
        "protocol": "five_fold_complete_matrix_cv", "dataset": "O'Neil", "grid": [4, 4],
        "random_state": RANDOM_STATE, "n_folds": 5,
        "validation_fraction_of_remaining": VAL_FRACTION_OF_REMAINING,
        "std_ddof": CV_STD_DDOF, "report_scope": "full_landscape_model_predictions",
        "aggregation": "repeat mean per matrix -> matrix mean per test fold -> mean/sample SD across five test folds",
        "model": MODEL_CONFIG, "train_observed_k_list": TRAIN_OBSERVED_K_LIST,
        "active_ensemble_size": ACTIVE_ENSEMBLE_SIZE,
        "fold_ensemble_seeds": {str(f): [int(seed) + f * 1009 for seed in ACTIVE_ENSEMBLE_SEEDS] for f in range(1, 6)},
        "initial_observations": ACTIVE_INITIAL_SEED_MODE, "acquisition_batch_size": ACTIVE_BATCH_SIZE,
        "acquisition_repeats": ACTIVE_REPEATS, "test_budgets": ACTIVE_TOTAL_BUDGETS,
        "drug_swap_augmentation": DRUG_SWAP_AUGMENTATION, "drug_swap_probability": DRUG_SWAP_PROB,
        "drug_swap_scope": "training only", "test_time_parameter_optimization": False,
        "crispr_top_n": CRISPR_TOP_N, "crispr_min_valid_fraction": CRISPR_MIN_VALID_FRACTION,
        "preprocessing_scope": "fit separately on unique training cells/drugs in each fold; validation/test transform only",
        "loss_weights": {"observed": LAMBDA_OBS_RECON, "delta": LAMBDA_DELTA,
                         "residual_consistency": LAMBDA_RESIDUAL_CONSISTENCY},
        "hyperparameters": {"max_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
                            "weight_decay": WEIGHT_DECAY, "patience": PATIENCE, "dropout": DROPOUT},
        "hyperparameter_selection": "fixed paper values; no grid search implemented by this notebook",
        "inputs": {name: {"path": str(path), "sha256": file_sha256(path)} for name, path in
                   {"combination": COMBO_FILE, "raw_graphlet": DRUG_RAW_FEATURE_FILE,
                    "raw_crispr": CRISPR_RAW_FEATURE_FILE}.items()},
        "python": platform.python_version(),
        "packages": {name: package_version(name) for name in ["numpy", "pandas", "scikit-learn", "torch", "joblib"]},
    }
    (RESULT_DIR / "run_config.json").write_text(json.dumps(configuration, ensure_ascii=False, indent=2), encoding="utf-8")
    fold_tables = []
    for fold_id, train_idx, val_idx, test_idx in splits:
        fold_dir = RESULT_DIR / f"fold_{fold_id}"
        fold_dir.mkdir(exist_ok=False)
        raw_train = [blocks[int(index)] for index in train_idx]
        preprocessor = fit_fold_preprocessors(raw_train, raw_drug, raw_crispr)
        transformed, drug_table, cell_table = transform_fold_blocks(blocks, raw_drug, raw_crispr, preprocessor)
        save_fold_preprocessing(fold_dir, fold_id, preprocessor, drug_table, cell_table)
        fold_metrics = train_one_test_fold(
            [transformed[int(index)] for index in train_idx],
            [transformed[int(index)] for index in val_idx],
            [transformed[int(index)] for index in test_idx], fold_dir, fold_id,
        )
        fold_tables.append(fold_metrics)
        del transformed, preprocessor, drug_table, cell_table
    fold_metrics = pd.concat(fold_tables, ignore_index=True)
    numeric, paper_table = summarize_five_test_folds(fold_metrics)
    fold_metrics.to_csv(RESULT_DIR / "fivefold_test_fold_metrics.csv", index=False, encoding="utf-8-sig")
    numeric.to_csv(RESULT_DIR / "fivefold_summary_numeric.csv", index=False, encoding="utf-8-sig")
    paper_table.to_csv(RESULT_DIR / "fivefold_paper_table.csv", index=False, encoding="utf-8-sig")
    print("\nFive held-out test folds: mean ± sample SD (ddof=1)")
    print(paper_table.to_string(index=False))
    print("Results:", RESULT_DIR)
    return fold_metrics, numeric, paper_table


if __name__ == "__main__":
    fold_metrics, fivefold_summary, fivefold_paper_table = main()